# Query Expansion

**Query expansion** (also referred to as multi-query) is a vital **pre-retrieval optimisation** technique used in advanced RAG systems to overcome the inherent limitations of distance-based similarity searches. Because a single query embedding may only cover a small area of the embedding space, it risks missing semantically related documents that don't sit immediately adjacent to that specific query vector.

### **The Concept of Query Expansion**
The core idea is to use a Large Language Model (LLM) to "rewrite" the original user input into **multiple different versions**. By generating these different perspectives, the retrieval module can capture various facets and interpretations of the same question. This **diversifies the search terms**, which significantly increases the likelihood of retrieving a more comprehensive and relevant set of data points from the vector database.

*   **Example:** An initial query like *"Write an article about the best types of advanced RAG methods"* might be expanded into variations such as *"What are the most effective advanced RAG methods, and how can they be applied?"* or *"Can you provide an overview of the top advanced retrieval-augmented generation techniques?"*.

### **Implementation Details**
The implementation described in the sources is modular and integrated into the broader retrieval workflow:

*   **The `QueryExpansion` Class:** This class inherits from a `RAGStep` interface. It manages the interaction with an LLM (typically **GPT-4o-mini**) to generate the expanded questions.
*   **Prompt Engineering:** The system utilises a `QueryExpansionTemplate` which provides a zero-shot prompt. This template instructs the LLM to generate alternative questions separated by a specific token, such as **`#next-question#`**, which the system uses to parse the generated string back into a list of query objects.

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

In [ ]:
# Load OPENAI API KEY
api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key

In [ ]:
# Base RAGStep Interface

from abc import ABC, abstractmethod
from typing import Any


class RAGStep(ABC):
    """
    Abstract base class for all RAG pipeline steps.
    """

    @abstractmethod
    def run(self, input_data: Any) -> Any:
        pass

In [ ]:
# Query Expansion Prompt Template

class QueryExpansionTemplate:
    """
    Zero-shot prompt template for generating multiple
    semantically diverse rewrites of a user query.
    """

    DELIMITER = "#next-question#"

    @staticmethod
    def build_prompt(query: str, n_variations: int) -> str:
        return f"""
You are an expert search query rewriter for retrieval-augmented generation systems.

Rewrite the following user query into {n_variations} alternative versions.
Each version should reflect a different perspective or phrasing
while preserving the original intent.

Separate each rewritten query using the token:
{QueryExpansionTemplate.DELIMITER}

User query:
\"\"\"{query}\"\"\"
"""

In [ ]:
# QueryExpansion Step (LLM-powered)

from typing import List
from openai import OpenAI

class QueryExpansion(RAGStep):
    """
    Uses an LLM to generate multiple rewritten queries
    for improved recall during vector retrieval.
    """

    def __init__(
        self,
        model: str = "gpt-4o-mini",
        n_variations: int = 3,
        temperature: float = 0.3,
    ):
        self.model = model
        self.n_variations = n_variations
        self.temperature = temperature
        self.client = OpenAI()

    def run(self, query: str) -> List[str]:
        prompt = QueryExpansionTemplate.build_prompt(
            query=query,
            n_variations=self.n_variations,
        )

        completion = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "developer", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}],
            temperature=self.temperature,
        )

        chatmessage = completion.choices[0].message

        raw_text = chatmessage.content

        expanded_queries = [
            q.strip()
            for q in raw_text.split(QueryExpansionTemplate.DELIMITER)
            if q.strip()
        ]

        return expanded_queries

In [ ]:
from typing import List
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

class QueryExpander:
    """
    LLM-based query expansion for multi-query retrieval.
    """

    DELIMITER = "#next-question#"

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        n_variations: int = 3,
        temperature: float = 0.3,
    ):
        self.n_variations = n_variations
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        )

        self.prompt = PromptTemplate(
            input_variables=["query", "n", "delimiter"],
            template="""
You are an expert search query rewriter for retrieval-augmented generation systems.

Rewrite the following user query into {n} alternative versions.
Each version should reflect a different perspective or phrasing
while preserving the original intent.

Separate each rewritten query using the token:
{delimiter}

User query:
"{query}"
""",
        )

    def expand(self, query: str) -> List[str]:
        formatted_prompt = self.prompt.format(
            query=query,
            n=self.n_variations,
            delimiter=self.DELIMITER,
        )

        response = self.llm.invoke(formatted_prompt).content

        expanded_queries = [
            q.strip()
            for q in response.split(self.DELIMITER)
            if q.strip()
        ]

        # Always include original query
        return [query] + expanded_queries


In [ ]:
query_expander = QueryExpansion()

In [ ]:
expanded_queries = query_expander.run("What is the meaning of life?")

In [ ]:
expanded_queries

['1. What does it mean to have a purpose in life?',
 '2. How can we define the significance of existence?',
 "3. What is the philosophical interpretation of life's meaning?"]